In [3]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
import pickle

# Download required NLTK data (just in case)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 1. Load the dataset (Make sure spam.csv is in the same folder)
df = pd.read_csv('spam.csv', encoding='latin-1')

# 2. Clean the data
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'], inplace=True, errors='ignore')
df.rename(columns={'v1':'target','v2':'text'}, inplace=True)

encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])
df = df.drop_duplicates(keep='first')

# 3. Text Preprocessing Function
ps = PorterStemmer()
def transform_text(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)
    
    text = y[:]
    y.clear()
    
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
            
    text = y[:]
    y.clear()
    
    for i in text:
        y.append(ps.stem(i))
            
    return " ".join(y)

# Apply transformation
print("Processing text... (this might take a few seconds)")
df['transformed_text'] = df['text'].apply(transform_text)

# 4. Initialize Vectorizer and Model
tfidf = TfidfVectorizer(max_features=3000) 
model = MultinomialNB()

# 5. Vectorize the data (The 'Fit' step)
print("Training the model...")
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

# 6. Split and Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
model.fit(X_train, y_train)

# 7. Verify Accuracy
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")

# 8. Export the properly trained files
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(model, open('model.pkl', 'wb'))
print("Fresh vectorizer.pkl and model.pkl saved successfully!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Processing text... (this might take a few seconds)
Training the model...
Accuracy: 0.9710
Precision: 1.0000
Fresh vectorizer.pkl and model.pkl saved successfully!
